# Forecast without an earth fault – estimates V0 to V3

An **optional part** of groundfault: the harmonic fault current at a busbar from network data and – with a recorder at
the transformer – from undisturbed recordings of the transformer node, **without a staged fault test**. Four methods
side by side:

| Method | Data | Result |
|---|---|---|
| **V0** classical estimate | U_m, I_C, coil | an upper estimate above about the resonance of the busbar, none below it |
| **V1** resonance bound | as V0, an upper limit G_max of the damping | the order of magnitude at the resonance |
| **V2** network data | as V0, damping d of the coil controller, rating plate, S″_k, C_1, an assumed damping G of the MV network | an estimate with G and an envelope over a range of G |
| **V3** transformer node | as V2, undisturbed recordings of HV voltages, MV voltages and transformer currents in one device | an estimate with the damping of the MV network measured |

The orders of the zero sequence (3, 9, 15 …) have a source that arises only in the fault: without an earth fault there
is no forecast for them. With staged fault tests or recorded earth faults use `network_model_forecast.ipynb` (V4 to
V7). Reference copy: `templates/forecast_without_fault_DEFAULT.ipynb`. Methods: online documentation, *Forecast*.

1. **Configuration** – network data, harmonic voltages, assumptions, recordings of the transformer node, limits.
2. **Network data and resonance** – upstream impedance, operating capacitance, resonance order m_r and m/m_r.
3. **V3** – the MV network from each undisturbed recording with its checks, and the pooled estimate of the reliable ones.
4. **V0 to V3 side by side** per order and fault impedance, and I_H.
5. **Export** of the tables to the forecast folder of the work folder.

In [ ]:
%matplotlib inline
import datetime
import sys
from pathlib import Path

LANGUAGE = "en"                # output language of tables, hints and figures: "en" | "de"
try:
    import groundfault as gf
except ImportError:            # not installed: use the source folder of a clone (…/groundfault/src)
    for p in [Path.cwd(), *Path.cwd().parents]:    # ./src, ./groundfault/src, ./*/groundfault/src
        try:
            candidates = [p / "src", p / "groundfault" / "src", *sorted(p.glob("*/groundfault/src"))]
        except OSError:
            candidates = [p / "src", p / "groundfault" / "src"]
        hits = [c for c in candidates if (c / "groundfault" / "__init__.py").is_file()]
        if hits:
            sys.path.insert(0, str(hits[0]))
            break
    import groundfault as gf
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from groundfault import forecast as fc
from groundfault import (campaign, corrections, demo, diagnostics, disturbances, document, earthing,
                         equivalent_source, explorer, i18n, interactive, layout, legacy, pq, register, report, rms, sites,
                         transformer)

i18n.set_language(LANGUAGE)
# work folder = this folder or the first folder above it with a campaigns folder ("campaigns" or
# "01_Messkampagnen"); its name gives the folder layout, without one the layout of LANGUAGE is used
WORK_FOLDER, LAYOUT = layout.find_work_folder(Path.cwd(), default=LANGUAGE)
WORK_FOLDER = Path(WORK_FOLDER)
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda x: "%.4g" % x)
report.activate_style()        # notebook preview = typeface of the exported figures


def show(df):
    """Table with column names and codes in the output language."""
    display(i18n.localize(df))


print("groundfault %s · work folder: %s · layout: %s · font of the figures: %s"
      % (gf.__version__, WORK_FOLDER, LAYOUT.language, report.used_font()))

## 1 · Configuration

In [ ]:
NETWORK = "<substation>"        # name of the network (substation, busbar) – also the sub-folder of the forecast folder
DEMO    = False

# ---------------------------------------------------------------- network data
U_N    = None                   # nominal voltage of the MV network in V
U_LE   = None                   # line-to-earth voltage (fundamental) in V
I_C    = None                   # capacitive earth-fault current in A (coil controller)
I_L    = None                   # coil current in A (coil controller; None = I_C, tuned)
D      = None                   # damping I_w/I_C (coil controller): series resistance d·X_C0 of the zero sequence in V2, V3
U_K    = None                   # short-circuit voltage of the transformer (e.g. 0.14 for 14 %)
S_RT   = None                   # rated power of the transformer in VA
S_K    = None                   # short-circuit power S″_k of the HV network in VA
C_1    = None                   # operating capacitance of the MV network per phase in F (cable lengths); None = Q_C·C_E
Q_C    = None                   # C_1/C_E when C_1 is None: cable networks about 1, mixed networks somewhat more
R_X_UP = None                   # R/X of the upstream impedance (transformer and HV network) at the orders – your assumption
R_X_HV = None                   # R/X of the HV network (corrected instrument of V3) – your assumption

# ---------------------------------------------------------------- harmonic voltages and fault impedances
ORDERS   = (5, 7)               # orders of the positive and negative sequence (not 3, 9, 15 …: their source arises in the fault)
VOLTAGES = None                 # harmonic voltages per order in V, e.g. {5: U_5, 7: U_7} (None = no forecast)
Z_F      = (0.0,)               # fault impedances in Ω
PQ_FILES    = None              # export(s) of a PQ instrument at the busbar → VOLTAGES (quantile × factor); relative to the work folder or absolute
PQ_TIMEZONE = None              # time zone of the instrument clock, e.g. "Europe/Berlin" (None = time stamps as exported)
PQ_QUANTILE = 0.95              # quantile of the 10-min values

# ---------------------------------------------------------------- assumptions for V1 and V2
G_MAX   = None                  # V1: upper limit of the damping of the MV network in S (None = V1 not computed)
G       = None                  # V2: assumed damping in S (None = no V2 estimate)
G_RANGE = None                  # V2: range of the damping in S for the envelope, (G_min, G_max) (None = no envelope)

# ---------------------------------------------------------------- V3: undisturbed recordings of the transformer node
RECORDINGS = []                 # pattern relative to the work folder (e.g. "campaigns/<campaign>/raw/<folder>/*.cfg") or a list of .cfg paths
SIGN       = None               # polarity of the transformer currents: +1 as measured, −1 reversed (None = from all recordings)
CT_ANGLE   = None               # angle error of the transformer current transformers in degrees: one value or {order: angle}
REFERENCE  = None               # reference clamp: dict(recording=…, channel=<transformer current>, reference=<clamp>) – gives
                                # the angle error per order; None = CT_ANGLE

# ---------------------------------------------------------------- limits – defined by you
# groundfault contains no limits. Not set (the values below), the notebook lists the values without assessing them.
MIN_HV_SHARE  = None            # V3: smallest share of the HV-driven variation of a reliable recording (None = not assessed)
V0_RATIO      = None            # V0: below m/m_r < V0_RATIO the column V0_rule_A doubles V0 (None = no such column)
FUNDAMENTAL_A = None            # fundamental residual current in A for I_RES (None = I_H only)
LIMITS        = ()              # limits of the residual current in A, e.g. (I_1, I_2)
PQ_SAFETY_FACTOR = 1.0          # factor on the quantile of the PQ measurement (1.0 = none)
PQ_FACTORS    = None            # factor on the quantile by the duration of the PQ measurement {minimum days: factor} (None = PQ_SAFETY_FACTOR)

## 2 · Network data and resonance

From the rating plate and the short-circuit power: X_T = u_k·U_n²/S_rT, X_Q = 1.1·U_n²/S″_k, the upstream impedance
Z_up,m = m·(X_T + X_Q)·(R/X + j) and the impedance of the HV network Z_HV,m = m·X_Q·(R/X + j) (for the corrected
instrument of V3). The earth capacitance C_E = I_C/(3·ω₁·U_LE) and the operating capacitance C_1 give the parallel
resonance of the busbar, m_r = √(X_C1/X_up) with X_C1 = 1/(ω₁·C_1). Above about m_r the positive and negative
sequence are capacitive in the fault loop and the classical estimate V0 lies above the current; below m_r they are
inductive, partly cancel the capacitive zero sequence, and the current can exceed V0 severalfold.

In [ ]:
X_T = fc.transformer_reactance(U_K, S_RT, U_N) if None not in (U_K, S_RT, U_N) else None
X_Q = fc.hv_network_reactance(S_K, U_N) if None not in (S_K, U_N) else None
X_UP = X_T + X_Q if None not in (X_T, X_Q) else None
C_E = fc.earth_capacitance(I_C, U_LE) if None not in (I_C, U_LE) else None
C1 = C_1 if C_1 is not None else (Q_C * C_E if None not in (Q_C, C_E) else None)
M_R = fc.resonance_order(X_UP, C1) if X_UP and C1 else None
Z_UP = {m: fc.at_order(X_UP, m, R_X_UP) for m in ORDERS} if None not in (X_UP, R_X_UP) else None
Z_HV = {m: fc.at_order(X_Q, m, R_X_HV) for m in ORDERS} if None not in (X_Q, R_X_HV) else None
NETWORK_DATA = pd.DataFrame([dict(X_T_Ohm=X_T, X_Q_Ohm=X_Q, X_up_Ohm=X_UP, C_E_uF=C_E * 1e6 if C_E else None,
                                  C_1_uF=C1 * 1e6 if C1 else None, m_r=M_R)])
show(NETWORK_DATA)
if M_R:
    show(pd.DataFrame([dict(order=m, f_Hz=50.0 * m, m_over_mr=m / M_R) for m in ORDERS]))
NOT_SET = [n for n, v in (("U_N", U_N), ("U_LE", U_LE), ("I_C", I_C), ("U_K", U_K), ("S_RT", S_RT), ("S_K", S_K),
                          ("R_X_UP", R_X_UP), ("R_X_HV", R_X_HV)) if v is None] + ([] if C1 else ["C_1 or Q_C"])
if NOT_SET:
    print("ℹ Not set (section 1): " + ", ".join(NOT_SET) + " – the methods that need them are left out.")

## 3 · V3 – the MV network from undisturbed recordings

In undisturbed operation the harmonic sources of the HV and of the MV network vary all the time. A variation driven
from the HV side changes busbar voltage and transformer current by ΔU = Z_MV·ΔI, one driven from the MV side by
ΔU = −Z_up·ΔI. The HV voltage, corrected by the drop over the HV network (E = U_HV + Z_HV·I_T), follows only the HV
sources and serves as **instrument**: per 200-ms window the deviations from the mean give
Z_MV = ΣΔU·ΔE*/ΣΔI·ΔE* in the natural sequence of each order. Before that the angle error of the transformer current
transformers is corrected (reference clamp `REFERENCE` or `CT_ANGLE`).

Checks per recording: correlation between current and instrument (`corr_I_hv`), a capacitive result with a positive
real part, and the **share of the HV-driven variation** in the variance of the busbar voltage (`hv_share`). The share
is what decides: where the MV side drives most of the variation the estimate drifts towards −Z_up, and neither the
correlation nor the angle shows it. The share changes over the day (e.g. higher at night), so record at several times.
The recordings that pass are pooled – the estimate for the forecast.

In [ ]:
def path_in_work_folder(p):
    p = Path(p)
    return p if p.is_absolute() else WORK_FOLDER / p


if DEMO:                         # demo: a measuring day – six recordings of one minute, the share of the HV side falls towards noon
    folder = path_in_work_folder(Path(RECORDINGS).parent)
    folder.mkdir(parents=True, exist_ok=True)
    z_mv = {m: 1 / complex(g, m * 2 * np.pi * 50.0 * 65e-6) for m, g in ((5, 0.07), (7, 0.09))}   # the truth of the demo
    for k, (hour, hv, mv) in enumerate(((2, 0.10, 0.05), (6, 0.08, 0.07), (10, 0.04, 0.10), (12, 0.02, 0.13),
                                        (16, 0.05, 0.09), (22, 0.09, 0.06))):
        cfg = folder / ("TRU_%02dh_10_000kHz.cfg" % hour)
        if not cfg.exists():
            demo.make_undisturbed_transformer_recording(str(cfg), duration=60.0, Z_mv=z_mv, hv_fluctuation=hv,
                                                        mv_fluctuation=mv, ct_angle_deg={5: 2.0, 7: 2.8},
                                                        reference_clamp=(k == 0), seed=30 + k,
                                                        start=datetime.datetime(2026, 3, 10, hour, 0))
FILES = (sorted(WORK_FOLDER.glob(RECORDINGS)) if isinstance(RECORDINGS, str)
         else [path_in_work_folder(p) for p in RECORDINGS])
CT_TABLE = V3_TABLE = V3_POOLED = None
Z_MV = {}
CT = CT_ANGLE
if REFERENCE:
    CT_TABLE = transformer.ct_angle_from_reference(str(path_in_work_folder(REFERENCE["recording"])),
                                                   REFERENCE["channel"], REFERENCE["reference"], orders=ORDERS)
    display(Markdown("**Angle error of the current transformers** against the reference clamp"))
    show(CT_TABLE)
    CT = CT_TABLE
if not FILES:
    print("ℹ V3 skipped – no recordings (RECORDINGS, section 1).")
elif Z_UP is None or Z_HV is None:
    print("ℹ V3 skipped – the upstream impedance needs U_N, U_K, S_RT, S_K, R_X_UP and R_X_HV (section 1).")
else:
    V3_TABLE, V3_POOLED = transformer.mv_network_series([str(f) for f in FILES], ORDERS, sign=SIGN, Z_hv=Z_HV,
                                                        Z_up=Z_UP, ct_angle_deg=CT, min_hv_share=MIN_HV_SHARE)
    cols = ["recording", "start", "order", "sequence", "Z_Ohm", "Z_angle_deg", "G_S", "B_S", "corr_I_hv", "hv_share",
            "reliable"]
    display(Markdown("**Per recording** (natural sequence; polarity of the currents: %+g)" % V3_TABLE.attrs["sign"]))
    show(V3_TABLE[cols])
    for name, hints in V3_TABLE.attrs["hints"].items():
        for h in hints:
            print("ℹ %s: %s" % (name, h))
    display(Markdown("**Pooled estimate of the reliable recordings**"))
    show(V3_POOLED[["order", "sequence", "recordings", "Z_Ohm", "Z_angle_deg", "Z_low_Ohm", "Z_high_Ohm", "G_S", "B_S",
                    "hv_share", "names"]])
    Z_MV = {int(r.order): complex(r.R_Ohm, r.X_Ohm) for r in V3_POOLED.itertuples() if r.recordings}
    if MIN_HV_SHARE is None:
        print("ℹ The share of the HV-driven variation (hv_share) is listed, not assessed – MIN_HV_SHARE is not set (section 1). Recordings in which the MV side drives most of the variation are then pooled too.")

## 4 · V0 to V3 side by side

Per order and fault impedance: **V0** = 3·U_m/|Z_0,m + 3·Z_F| with the capacitive Z_0 from I_C and the coil;
**V1** = 3·U_m/|2/G_max + Z_0,m + 3·Z_F|; **V2** the full fault loop 3·U_m/|Z_0 + Z_1 + Z_2 + 3·Z_F| with
Z_1 = Z_2 = (1/Z_up + G + j·m·ω₁·C_1)⁻¹ and Z_0 with the series resistance d·X_C0 – the estimate with `G` and the
envelope (largest current) over `G_RANGE`, there without damping of the zero sequence; **V3** the same loop with the
MV network measured (section 3). With `V0_RATIO` the column `V0_rule_A` doubles V0 below m/m_r < `V0_RATIO`. The rows
`I_H` are √Σ I_m² over the orders of the table.

In [ ]:
COMPARISON = None
PQ = None
if PQ_FILES:                           # PQ measurement of the busbar: quantile per order (line-to-earth, largest phase) × factor
    files = [WORK_FOLDER / p for p in ([PQ_FILES] if isinstance(PQ_FILES, (str, Path)) else PQ_FILES)]
    PQ = pq.assess(pq.read(files, timezone=PQ_TIMEZONE), quantile=PQ_QUANTILE,
                   **(dict(factors=PQ_FACTORS) if PQ_FACTORS else dict(factor=PQ_SAFETY_FACTOR)))
    print(PQ.summary())
    for h in PQ.hints:
        print("ℹ", h)
    if PQ.valid:
        VOLTAGES = PQ.voltages() if PQ.reference == "LE" else {m: u / 3 ** 0.5 for m, u in PQ.voltages().items() if m % 3}
        print(i18n.format_text("VOLTAGES from the PQ measurement (quantile × {:g}, line-to-earth): {}", PQ.factor,
                               ", ".join(i18n.format_text("{}: {:.1f} V", m, VOLTAGES[m]) for m in ORDERS if m in VOLTAGES)))
    else:
        print("⚠ The PQ measurement is not valid for the assessment (gapless values shorter than every duration of PQ_FACTORS) – no voltages from it; VOLTAGES of section 1 stays.")
if VOLTAGES is None:
    print("ℹ No forecast – VOLTAGES is not set (section 1).")
elif None in (U_LE, I_C) or Z_UP is None:
    print("ℹ No forecast – network data missing (section 2).")
else:
    COMPARISON = fc.estimates_without_fault({m: VOLTAGES[m] for m in ORDERS if m in VOLTAGES}, I_C, U_LE, Z_UP,
                                            Z_F=tuple(Z_F), I_L=I_L, d=D or 0.0, C_1=C1, m_r=M_R, G_max=G_MAX,
                                            G=G, G_range=G_RANGE, Z_mv=Z_MV or None)
    if V0_RATIO is not None and M_R:
        rule = [v * (2.0 if isinstance(o, (int, np.integer)) and o / M_R < V0_RATIO else 1.0)
                for o, v in zip(COMPARISON.order, COMPARISON.V0_A)]
        COMPARISON.insert(list(COMPARISON.columns).index("V0_A") + 1, "V0_rule_A", rule)
        for zf in COMPARISON.Z_F_Ohm.unique():      # I_H of the rule: √Σ over the orders
            part = (COMPARISON.Z_F_Ohm == zf) & (COMPARISON.order != "I_H")
            COMPARISON.loc[(COMPARISON.Z_F_Ohm == zf) & (COMPARISON.order == "I_H"), "V0_rule_A"] = \
                float(np.sqrt((COMPARISON.loc[part, "V0_rule_A"] ** 2).sum()))
    show(COMPARISON)
    names = {"V0_A": "V0", "V0_rule_A": "V0 rule", "V1_A": "V1", "V2_A": "V2",
             "V2_envelope_A": "V2 envelope", "V3_A": "V3"}
    methods = [c for c in names if c in COMPARISON.columns]
    for zf in COMPARISON.Z_F_Ohm.unique():
        h = COMPARISON[(COMPARISON.Z_F_Ohm == zf) & (COMPARISON.order == "I_H")].iloc[0]
        text = ", ".join(i18n.format_text("{} {:.1f} A", names[c], h[c]) for c in methods)
        if FUNDAMENTAL_A is not None:
            text += " · I_RES: " + ", ".join(i18n.format_text("{} {:.1f} A", names[c], float(np.hypot(FUNDAMENTAL_A, h[c])))
                                             for c in methods)
        print(i18n.format_text("Z_F = {:g} Ω – I_H: ", zf) + text)
        if LIMITS and FUNDAMENTAL_A is not None:
            for limit in LIMITS:
                over = [names[c] for c in methods if np.hypot(FUNDAMENTAL_A, h[c]) > limit]
                print(i18n.format_text("   I_RES above {:g} A: ", limit) + (", ".join(over) or "–"))
    if FUNDAMENTAL_A is None:
        print("ℹ I_RES not computed – FUNDAMENTAL_A is not set (section 1).")

## 5 · Export the tables

In [ ]:
OUT = Path(fc.forecast_folder(WORK_FOLDER, NETWORK))
TABLES = {"forecast_network_data": NETWORK_DATA, "forecast_ct_angle": CT_TABLE, "forecast_v3_recordings": V3_TABLE,
          "forecast_v3": V3_POOLED, "forecast_without_fault": COMPARISON}
TABLES = {k: v for k, v in TABLES.items() if v is not None and len(v) and v.notna().any().any()}
if TABLES:
    paths = report.export_tables(TABLES, str(OUT), prefix=report.slug(NETWORK) + "_without_fault_",
                                 formats=("csv", "md"))
    print("%d table files in %s" % (len(paths), OUT))
else:
    print("ℹ Nothing to export – no network data, recordings or voltages (section 1).")